# Experiment

## Import libraries

In [9]:
import pandas as pd

file_path = "vietstock_2000_2025.csv"

df = pd.read_csv(file_path)

df = df.iloc[:5, :]

# Set indicator names as lowercase with underscores
df["Chỉ tiêu"] = df["Chỉ tiêu"].str.lower().str.replace(" ", "_")

# Melt from wide to long format
df = df.melt(
    id_vars=["Chỉ tiêu", "Đơn vị tính"], var_name="quarter_str", value_name="value"
)

# Filter out any non-quarter columns
df = df[df["quarter_str"].str.match(r"Q\d+/\d{4}")]

# Clean numeric values
df["value"] = df["value"].astype(str).str.replace(",", "", regex=False)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Extract year and quarter
df["quarter"] = df["quarter_str"].str.extract(r"Q(\d+)/")[0].astype(int)
df["year"] = df["quarter_str"].str.extract(r"/(\d{4})")[0].astype(int)

# Use pivot_table with first() to handle duplicates
df = df.pivot_table(
    index=["year", "quarter"], columns="Chỉ tiêu", values="value", aggfunc="first"
).reset_index()

# Sort by year and quarter
df = df.sort_values(["year", "quarter"]).reset_index(drop=True)

# Fill missing values with 0
df.fillna(0, inplace=True)

# Rename columns
df.rename(columns={"total_gdp": "gdp_growth"}, inplace=True)

# Resort columns
df = df[
    ["year", "quarter", "agriculture", "industry", "services", "gdp_growth", "gdp_real"]
]

df

Chỉ tiêu,year,quarter,agriculture,industry,services,gdp_growth,gdp_real
0,2005,1,0.0,0.0,0.0,7.20,0.0
1,2006,1,0.0,0.0,0.0,7.20,0.0
2,2007,1,0.0,0.0,0.0,7.70,0.0
3,2008,1,0.0,0.0,0.0,7.43,0.0
4,2008,2,0.0,0.0,0.0,5.70,0.0
...,...,...,...,...,...,...,...
58,2024,2,151303.0,545237.0,677030.0,6.93,1508387.0
59,2024,3,161767.0,588545.0,680490.0,7.43,1567923.0
60,2024,4,192991.0,657896.0,776143.0,7.55,1773652.0
61,2025,1,153001.0,526666.0,681303.0,6.93,1502807.0
